In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

### 1. Tool creation

In [10]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
    """Given two integers a and b, this tool returns their product"""
    
    return a * b

In [11]:
print(multiply.invoke({'a':3,'b':4}))

12


In [12]:
print(multiply.name)

print(multiply.description,"\n")
print(multiply.args)

multiply
Given two integers a and b, this tool returns their product 

{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


### 2. Tool binding -> binding 2 or more tools :

In [13]:
llm = ChatOpenAI()

In [14]:
llm.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CAwpOwJDOMvgGVwAmJ8E4QHzYMFAm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--f24f0283-f965-4f72-b50a-ad8f68e44b3f-0', usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
# bind tools
llm_with_tools = llm.bind_tools([multiply]) #  if 2 tools are there then add them to the list by , seperated

### NOTE : Every LLM doesnot have the functionality of tool binding

In [16]:
# no tool calling , just checking if llm is working fine
llm_with_tools.invoke('hi how are you')

AIMessage(content="I'm just a computer program, so I don't have feelings, but I'm here to help you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 58, 'total_tokens': 88, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CAwpPUr0jTyF0WQlwOvYOG2pvHHhU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--64c41867-21df-41fc-85ef-5e3fa5584a3d-0', usage_metadata={'input_tokens': 58, 'output_tokens': 30, 'total_tokens': 88, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [17]:
query = HumanMessage('can you multiply 3 with 10')

In [18]:
message = [query]

In [19]:
message

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={})]

#### another example

In [ ]:
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# 1. Define tools
@tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

tools = [add_numbers, multiply_numbers]

# 2. Define LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# 3. Bind tools
llm_with_tools = llm.bind_tools(tools) # tool binding is happening here

# 4. Assistant node (LLM may decide to call tools)
def assistant(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# 5. Tool execution node
tool_node = ToolNode(tools)

# 6. Build graph
graph = StateGraph(MessagesState)
graph.add_node("assistant", assistant)
graph.add_node("tools", tool_node)

# edges: assistant → tools → assistant
graph.add_edge("assistant", "tools")
graph.add_edge("tools", "assistant")

graph.set_entry_point("assistant")

# 7. Compile
app = graph.compile()

# 8. Run
print("\n--- Tool Binding Example ---")
res1 = app.invoke({"messages": [("user", "What is 5 plus 7?")]})
print("Final:", res1["messages"][-1].content)

res2 = app.invoke({"messages": [("user", "Multiply 9 and 6")]})
print("Final:", res2["messages"][-1].content)


### 3. Tool calling

In [20]:
# tool call
result = llm_with_tools.invoke(message)
result


AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_kq0p56rCijXoY91XJBgTKABP', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 62, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CAwpRkzxoZKiVRqV8SZbbj6ZRPOjG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--c991aea9-ddcc-4bf6-8e16-de36c8620ee3-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_kq0p56rCijXoY91XJBgTKABP', 'type': 'tool_call'}], usage_metadata={'input_tokens': 62, 'output_tokens': 17, 'total_tokens': 79, 'input_token_details': {'audio': 0, 'cache_read': 0}, 

In [30]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'call_kq0p56rCijXoY91XJBgTKABP',
 'type': 'tool_call'}

In [31]:
result.tool_calls[0]['name']

'multiply'

In [23]:
message.append(result)

In [24]:
message
# it has both human and AI message

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_kq0p56rCijXoY91XJBgTKABP', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 62, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CAwpRkzxoZKiVRqV8SZbbj6ZRPOjG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--c991aea9-ddcc-4bf6-8e16-de36c8620ee3-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_kq0p56rCijXoY91XJBgTKABP', 'type': 'tool_call'}], usage_metadata={'input_tokens': 6


### NOTE : LLM can never call the tool, it can only suggest the tool

In [25]:
# passing the iput of llm to the tool
multiply.invoke(
    {'name': 'multiply',
     'args': {'a': 3, 'b': 10},
     'id': 'call_U9VWkwh2gmuqtQCJji0J6rQO',
     'type': 'tool_call'}
)

ToolMessage(content='30', name='multiply', tool_call_id='call_U9VWkwh2gmuqtQCJji0J6rQO')

#### Tool Message is a special message given by tool call which can be sent to the LLM

### 4. Tool Execution

In [32]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 3, 'b': 10},
 'id': 'call_kq0p56rCijXoY91XJBgTKABP',
 'type': 'tool_call'}

In [26]:
# tool execution
tool_result = multiply.invoke(result.tool_calls[0]) # instead of the arguments we are sending entire tool call
tool_result

ToolMessage(content='30', name='multiply', tool_call_id='call_kq0p56rCijXoY91XJBgTKABP')

In [27]:
message.append(tool_result) # appending tool_result in the message list

In [28]:
message

# has 3 things : 
# 1. Human message
# 2. Ai messgae
# 3. Tool message

[HumanMessage(content='can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_kq0p56rCijXoY91XJBgTKABP', 'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 62, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CAwpRkzxoZKiVRqV8SZbbj6ZRPOjG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--c991aea9-ddcc-4bf6-8e16-de36c8620ee3-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_kq0p56rCijXoY91XJBgTKABP', 'type': 'tool_call'}], usage_metadata={'input_tokens': 6

### Final output through LLM

In [29]:
llm_with_tools.invoke(message)

AIMessage(content='The product of 3 and 10 is 30.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 87, 'total_tokens': 100, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CAwpScNAwFvNKrmfsArD1td1ph3n2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--ec5aa021-638f-45c8-81d2-ae5a6c255309-0', usage_metadata={'input_tokens': 87, 'output_tokens': 13, 'total_tokens': 100, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Tool Node


### 🔹 What is `ToolNode`?

A **`ToolNode`** is a special LangGraph node that acts as a **bridge between the LLM and your Python tools**.

* The LLM itself **does not execute Python functions**.


* Instead, it produces a **tool call request** (a JSON payload saying: *“call tool X with args Y”*).


* The `ToolNode` listens for those requests, **executes the corresponding Python function**, and injects the result back into the graph’s state.

Think of it as the **executor/dispatcher** for tool calls.





#### 🔹 Why do we need it?

* Without a `ToolNode`, your assistant node (LLM) can *say* it wants to use a tool, but nothing will actually run.


* `ToolNode` ensures that when the LLM outputs a tool call, the right Python function is triggered and the result is returned.





### 🔹 When to Use `ToolNode`

✅ Use `ToolNode` when:

* You’re **manually building a graph** with `StateGraph`.


* You want fine-grained control over how nodes and tools interact.


* You’re building custom workflows, not just simple agents.





### 🔹 When NOT to Use `ToolNode`

❌ Don’t use `ToolNode` when:

* You use **prebuilt agent constructors** like `create_react_agent`, `create_openai_functions_agent`, etc.
* These already include the logic to:

  1. Let the LLM decide when to call a tool.
  2. Dispatch tool calls.
  3. Return results to the LLM.

So adding a `ToolNode` there would be redundant.




### 🔹 Example 1: Using `ToolNode` (manual graph)

```python
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Define tools
@tool
def add(a: int, b: int) -> int:
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    return a * b

tools = [add, multiply]

# LLM with tool awareness
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

# Assistant node
def assistant(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# Tool executor node
tool_node = ToolNode(tools)

# Build graph manually
graph = StateGraph(MessagesState)
graph.add_node("assistant", assistant)
graph.add_node("tools", tool_node)
graph.add_edge("assistant", "tools")
graph.add_edge("tools", "assistant")
graph.set_entry_point("assistant")

app = graph.compile()

# Run
result = app.invoke({"messages": [("user", "Multiply 8 and 7")]})
print(result["messages"][-1].content)
```

Here:

* Assistant decides: “I should call `multiply`”.
* `ToolNode` executes `multiply(8,7)`.
* Assistant gets result and responds.





#### 🔹 Example 2: Without `ToolNode` (prebuilt agent)

```python
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Tools
@tool
def add(a: int, b: int) -> int:
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    return a * b

tools = [add, multiply]

# LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Prebuilt agent (no ToolNode needed)
agent = create_react_agent(llm, tools)

# Run
result = agent.invoke({"messages": [("user", "Add 12 and 6")]})
print(result["messages"][-1].content)
```

Here:

* You never add `ToolNode`.


* The `create_react_agent` already bundles the `assistant + tool dispatcher` logic internally.



### 🔑 Summary

* **ToolNode = dispatcher** that actually executes tools when building **custom graphs**.


* **Use it** if: you’re building workflows manually with `StateGraph`.


* **Don’t use it** if: you’re using prebuilt agent creators (`create_react_agent`, etc.), since they handle tool execution internally.
